# The Anatomy of AI · Lecture 13 · Operating on a Painter — the notebook

So far we have read neurons. Here we operate on them, inside a network that paints churches (StyleGAN2, trained on
photos of churches). You will:
1. paint churches from random numbers;
2. **name** the painter's neurons by comparing their maps with a segmenter;
3. **switch the tree neurons off** and watch the trees go, on one church and then on 200;
4. **switch them on** in one small patch, in the sky and on a wall;
5. look for door neurons, and find a stock-photo watermark the painter remembered.

Each section names the lecture plate it follows. Wherever the lecture gives a number, a `check(...)` line compares
yours with it and prints ✓; if a check fails, the notebook stops.

**Runs on:** free Colab with a T4 GPU, in about fifteen minutes. It downloads three models: NVIDIA's StyleGAN2 church
model (357 MB), and two segmenters from Hugging Face, SegFormer-B4 (250 MB) and Mask2Former Swin-L (860 MB).

## 0. Setup
Installs (pinned, including NVIDIA's StyleGAN code at the commit the lecture used), the plot style and `check()`.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys, os, io, zipfile
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "requests==2.32.3", "scipy==1.14.1", "click==8.1.7",
    "transformers==4.44.2", "safetensors==0.4.5", "huggingface_hub==0.25.2", "setuptools==75.6.0")
# NVIDIA's StyleGAN code (its loader reads the original 2019 model file), at the lecture's commit, from GitHub's zip
import requests
COMMIT = "c233a919a6faee6e36a316ddd4eddababad1adf9"
if not os.path.exists("stylegan3"):
    z = zipfile.ZipFile(io.BytesIO(requests.get(f"https://github.com/NVlabs/stylegan3/archive/{COMMIT}.zip", timeout=120).content))
    z.extractall("."); os.rename(f"stylegan3-{COMMIT}", "stylegan3")
sys.path.insert(0, "stylegan3")
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
print("installed; stylegan3 at", COMMIT[:7])

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

In [ ]:
import contextlib, pickle
import setuptools                                  # Python 3.12 dropped distutils; setuptools stands in for the StyleGAN code
import legacy                                      # from the stylegan3 code
from torch_utils.ops import bias_act, upfirdn2d
bias_act._init = lambda: False                     # no compiler needed: use the code's plain PyTorch versions
upfirdn2d._init = lambda: False
PKL = "https://nvlabs-fi-cdn.nvidia.com/stylegan2/networks/stylegan2-church-config-f.pkl"
if not os.path.exists("church.pkl"):
    with requests.get(PKL, stream=True, timeout=300) as r:
        r.raise_for_status()
        with open("church.pkl.part", "wb") as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk)
    os.rename("church.pkl.part", "church.pkl")
with open("church.pkl", "rb") as f, contextlib.redirect_stderr(io.StringIO()), contextlib.redirect_stdout(io.StringIO()):
    G = legacy.load_network_pkl(f)["G_ema"].eval().requires_grad_(False).to(DEV)
print(f"StyleGAN2 church model ({os.path.getsize('church.pkl') / 1e6:.0f} MB): ready")

mods = dict(G.synthesis.named_modules())
feats, edits = {}, {}
for L in ("b16.conv1", "b32.conv1"):
    mods[L].register_forward_hook(lambda m, i, o, L=L: edits[L](o) if L in edits else feats.__setitem__(L, o))

def paint(seeds, edit=None, layer="b32.conv1"):
    """Churches from these seeds (each seed -> 512 random numbers -> a picture), optionally editing a layer on the way."""
    if edit is not None:
        edits[layer] = lambda o: (feats.__setitem__(layer, o), edit(o))[1]
    z = torch.from_numpy(np.stack([np.random.RandomState(s).randn(G.z_dim) for s in seeds])).float().to(DEV)
    with torch.no_grad():
        img = G(z, None, truncation_psi=0.8, noise_mode="const")
    edits.clear()
    return (img.clamp(-1, 1) + 1) / 2                 # B x 3 x 256 x 256 in [0, 1]

def zero(units):
    idx = torch.tensor(units, device=DEV, dtype=torch.long)
    def f(o):
        o = o.clone(); o[:, idx] = 0
        return o
    return f

show = lambda x: x.permute(1, 2, 0).cpu().numpy()
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False   # the same answer on every run

## 1. A network that paints (Plate I)
None of these churches exists. Each starts as a *seed*: 512 random numbers. The generator turns them into a picture block
by block, 4 pixels across, then 8, 16, … 256. It is a *GAN*: it learned by trying to fool a critic that tells its
pictures from real photos.

In [ ]:
check("random numbers per church", G.z_dim, 512)
S = G.synthesis
print("blocks:", S.block_resolutions)
for r in S.block_resolutions:
    print(f"  b{r}: {getattr(S, f'b{r}').conv1.weight.shape[0]} neurons")
check("blocks, by picture size", list(S.block_resolutions), [4, 8, 16, 32, 64, 128, 256])
check("neurons per block", [getattr(S, f"b{r}").conv1.weight.shape[0] for r in S.block_resolutions], [512, 512, 512, 512, 512, 256, 128])
x = paint([0, 1, 2, 5018, 5164, 5019])
fig, axs = plt.subplots(1, 6, figsize=(11, 2.2))
for a, im, s in zip(axs, x, [0, 1, 2, 5018, 5164, 5019]):
    a.imshow(show(im)); a.axis("off"); a.set_title(f"seed {s}", fontsize=9)
plt.show()

## 2. Naming the neurons: dissection (Plates II–IV)
We open the painter at two blocks, b16 and b32. Each neuron there has a map over the canvas (16 × 16 or 32 × 32). To name
it we use a *segmenter*, a second network that names every pixel (tree, building, sky…): SegFormer-B4, trained on the
ADE20K scenes.

For each neuron and each name, compare two masks on 1,000 churches (seeds 0–999):
- "the neuron is on": above its 99th or 95th percentile, whichever fits better;
- "the segmenter says tree": more than half the pixels of that cell.

Their overlap is the *IoU*: places in both ÷ places in either. This is Lecture 9's Network Dissection, run on a painter.

In [ ]:
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor, logging as hf_logging
hf_logging.set_verbosity_error()
seg_name = "nvidia/segformer-b4-finetuned-ade-512-512"
with contextlib.redirect_stderr(io.StringIO()):
    sproc = SegformerImageProcessor.from_pretrained(seg_name)
    sf = SegformerForSemanticSegmentation.from_pretrained(seg_name).to(DEV).eval()
CLS = {v: int(k) for k, v in sf.config.id2label.items()}
print("SegFormer-B4: ready; classes used:", {w: CLS[w] for w in ("tree", "building", "sky", "door")})

def segformer(img):
    x = F.interpolate(img, size=(512, 512), mode="bilinear", align_corners=False)
    mean = torch.tensor(sproc.image_mean, device=DEV)[None, :, None, None]; std = torch.tensor(sproc.image_std, device=DEV)[None, :, None, None]
    with torch.no_grad():
        lg = sf(pixel_values=(x - mean) / std).logits
    return F.interpolate(lg, size=(256, 256), mode="bilinear", align_corners=False).argmax(1)

N = 1000
acts = {L: [] for L in ("b16.conv1", "b32.conv1")}; segs = []
t = time.time()
for s0 in range(0, N, 25):
    img = paint(range(s0, s0 + 25))
    segs.append(segformer(img).to(torch.uint8).cpu())
    for L in acts:
        acts[L].append(feats[L].half().cpu())
segs = torch.cat(segs)
print(f"1,000 churches painted and segmented in {time.time() - t:.0f} s")
SHARE = {"tree": 0.089639, "door": 3.7e-05, "building": 0.42538, "sky": 0.362327}
for w in ("tree", "building", "sky", "door"):
    check(f"share of all pixels the segmenter calls {w}", float((segs == CLS[w]).float().mean()), SHARE[w], tol=max(0.00002, 0.01 * SHARE[w]))

In [ ]:
def dissect(L, names=("tree", "door", "sky")):
    A = torch.cat(acts[L])                                    # N x 512 x r x r (half)
    r = A.shape[-1]
    step = max(1, N * r * r // 20000)                         # the percentiles from about 20,000 places per neuron
    q99, q95 = [], []
    for i in range(0, 512, 64):
        flat = A[:, i:i + 64].float().permute(1, 0, 2, 3).reshape(64, -1)[:, ::step]
        q99.append(torch.quantile(flat, 0.99, dim=1)); q95.append(torch.quantile(flat, 0.95, dim=1))
    q99, q95 = torch.cat(q99), torch.cat(q95)
    out = {"q99": q99}
    for w in names:
        mask = F.adaptive_avg_pool2d((segs == CLS[w]).float()[:, None], r)[:, 0] > 0.5
        best = torch.zeros(512)
        for q in (q99, q95):
            inter = torch.zeros(512); uni = torch.zeros(512)
            for s in range(0, N, 100):
                on = A[s:s + 100].float() > q[None, :, None, None]
                m_ = mask[s:s + 100, None]
                inter += (on & m_).sum((0, 2, 3)).float(); uni += (on | m_).sum((0, 2, 3)).float()
            best = torch.maximum(best, inter / uni.clamp(min=1))
        o = torch.argsort(-best)
        out[w] = [(int(u), float(best[u])) for u in o[:40]]
        out[w + "_n05"] = int((best > 0.05).sum())
    return out

D32, D16 = dissect("b32.conv1"), dissect("b16.conv1")
print("b32's best tree neurons:", [(u, round(v, 3)) for u, v in D32["tree"][:8]])
print("b16's best tree neurons:", [(u, round(v, 3)) for u, v in D16["tree"][:3]])
for (u, v), (u2, v2) in zip(D32["tree"][:3], [[287, 0.436], [273, 0.383], [140, 0.346]]):
    check(f"b32: tree neuron no. {D32['tree'].index((u, v)) + 1}", u, u2)
    check(f"  its IoU with trees", v, v2, tol=0.002)
check("b32: neurons with tree IoU over 0.05", D32["tree_n05"], 71)
check("b32: the 10 best tree neurons, in order", [u for u, _ in D32["tree"][:10]], [287, 273, 140, 39, 403, 375, 433, 245, 177, 204, 230, 364, 395, 282, 406, 447, 281, 163, 417, 59, 443, 142, 33, 272, 51, 369, 75, 218, 132, 151, 314, 14, 495, 126, 229, 65, 300, 467, 55, 120][:10])
check("b32: of the lecture's 40 best, how many are in ours", len(set(u for u, _ in D32["tree"]) & set([287, 273, 140, 39, 403, 375, 433, 245, 177, 204, 230, 364, 395, 282, 406, 447, 281, 163, 417, 59, 443, 142, 33, 272, 51, 369, 75, 218, 132, 151, 314, 14, 495, 126, 229, 65, 300, 467, 55, 120])), 40, tol=2)
check("b16: best tree IoU", D16["tree"][0][1], [59, 0.307][1], tol=0.002)
check("b16: neurons with tree IoU over 0.05", D16["tree_n05"], 59)
TREE = [u for u, _ in D32["tree"]]; Q99 = D32["q99"].to(DEV)

In [ ]:
# What dissection sees on church 5018: the segmenter's labels, and where the best tree neuron (b32:287) is on
img = paint([5018]); lab = segformer(img)[0].cpu().numpy(); a = feats["b32.conv1"][0, TREE[0]].cpu()
on = (a > Q99[TREE[0]].cpu()).float()
fig, axs = plt.subplots(1, 3, figsize=(9.5, 3.3))
axs[0].imshow(show(img[0])); axs[0].set_title("church 5018", fontsize=9)
grey = np.full(lab.shape, 0.85); grey[lab == CLS["tree"]] = 0.25; grey[lab == CLS["building"]] = 0.7; grey[lab == CLS["sky"]] = 0.97
axs[1].imshow(grey, cmap="gray", vmin=0, vmax=1); axs[1].set_title("the segmenter: tree dark, building grey, sky light", fontsize=9)
up = F.interpolate(on[None, None], size=256)[0, 0].numpy() > 0.5      # the neuron's on-cells, at picture size
pic = show(img[0]); comp = pic * 0.35 + 0.65; comp[up] = pic[up]            # full colour where it is on, washed elsewhere
axs[2].imshow(comp)
axs[2].set_title(f"where b32:{TREE[0]} is on", fontsize=9)
for ax_ in axs: ax_.axis("off")
plt.show()

The best tree neuron's places match the trees. b32 has the sharper tree neurons; section 4 compares what switching them off does in each block.

## 3. The operation (Plate V)
Switch the best tree neurons off, setting them to zero everywhere, and let the painter finish. To measure the trees we
use a second, stronger segmenter, Mask2Former, as the lecture did for single churches. As a control, switch off the same
number of *random* other neurons (a fixed shuffle, numpy seed 0).

In [ ]:
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation
m2_name = "facebook/mask2former-swin-large-ade-semantic"
with contextlib.redirect_stderr(io.StringIO()):
    mproc = AutoImageProcessor.from_pretrained(m2_name)
    m2f = Mask2FormerForUniversalSegmentation.from_pretrained(m2_name).to(DEV).eval()
MCLS = {v: int(k) for k, v in m2f.config.id2label.items()}
print("Mask2Former: ready")

def mask2former(img):
    ims = [Image.fromarray((show(x) * 255).astype("uint8")).resize((512, 512), Image.BICUBIC) for x in img]
    inp = mproc(images=ims, return_tensors="pt").to(DEV)
    with torch.no_grad():
        o = m2f(**inp)
    return torch.stack([m_.to(torch.uint8) for m_ in mproc.post_process_semantic_segmentation(o, target_sizes=[(256, 256)] * len(ims))]).cpu()

from PIL import Image
share = lambda sg, w, cls=MCLS: float((sg == cls[w]).float().mean())
rng = np.random.default_rng(0)
RAND = rng.permutation([u for u in range(512) if u not in TREE]).tolist()

C5018 = {"0": 0.4252, "4": 0.2868, "6": 0.2234, "8": 0.2455, "16": 0.0465, "40": 0.039}
pics = {}
for k in (0, 4, 6, 8, 16, 40):
    im = paint([5018], zero(TREE[:k]) if k else None); pics[k] = im[0]
    v = share(mask2former(im), "tree")
    print(f"  {k:2d} tree neurons off: tree {100 * v:.1f}% of the picture")
    check(f"church 5018, {k} tree neurons off: tree share", v, C5018[str(k)], tol=0.008)
im = paint([5018], zero(RAND[:40])); pics["random"] = im[0]
v = share(mask2former(im), "tree")
check("church 5018, 40 random neurons off: tree share", v, 0.4369, tol=0.008)
fig, axs = plt.subplots(1, 5, figsize=(11, 2.5))
for a, k in zip(axs, (0, 4, 16, 40, "random")):
    a.imshow(show(pics[k])); a.axis("off"); a.set_title(f"{k} tree neurons off" if k != "random" else "40 random off", fontsize=9)
plt.show()

The trees peel away, and the white church appears where they were. Random neurons change nothing. (Not every step goes down: 8 off leaves a little more than 6.) One church's share can move by half a point between GPUs, where the trees are half gone, so these checks allow ±0.8 points; the 200-church numbers below are steadier.

## 4. Over 200 churches, and what fills the hole (Plates VI–VII)
One church could be luck. Repeat on 200 new churches (seeds 5000–5199) and measure with SegFormer, as the lecture did.
The random control replays the lecture's draws (numpy seed 0, one draw for each number of neurons, in order).

In [ ]:
TEST = list(range(5000, 5200))
def shares(edit=None, layer="b32.conv1"):
    cnt = {w: 0.0 for w in ("tree", "building", "sky")}
    for i in range(0, 200, 25):
        sg = segformer(paint(TEST[i:i + 25], edit, layer))
        for w in cnt:
            cnt[w] += float((sg == CLS[w]).float().sum())
    return {w: v / (200 * 256 * 256) for w, v in cnt.items()}

AB = {"0": [0.0867, 0.424, 0.3599], "4": [0.0491, 0.4519, 0.369], "16": [0.0255, 0.4698, 0.3808], "32": [0.0173, 0.4794, 0.3795], "40": [0.0192, 0.4814, 0.3786]}
others = [u for u in range(512) if u not in TREE]
rng = np.random.default_rng(0)
rand_sets = {k: sorted(rng.choice(others, k, replace=False).tolist()) if k else [] for k in (0, 1, 2, 4, 8, 12, 16, 24, 32, 40)}
t = time.time()
res = {k: shares(zero(TREE[:k]) if k else None) for k in (0, 4, 16, 32, 40)}
rnd40 = shares(zero(rand_sets[40]))
print(f"200 churches x 6 ways in {time.time() - t:.0f} s")
base = res[0]["tree"]
for k, r in res.items():
    print(f"  {k:2d} off: trees left {100 * r['tree'] / base:5.1f}%, building {100 * r['building']:.1f}%, sky {100 * r['sky']:.1f}%")
    check(f"{k} tree neurons off: tree share", r["tree"], AB[str(k)][0], tol=0.0006)
check("40 random neurons off: tree share", rnd40["tree"], 0.0895, tol=0.0006)
check("trees left with 32 off (%)", 100 * res[32]["tree"] / base, round(100 * AB["32"][0] / AB["0"][0], 1), tol=1)
check("building, before", res[0]["building"], AB["0"][1], tol=0.001)
check("building, 40 off", res[40]["building"], AB["40"][1], tol=0.001)
check("sky, before", res[0]["sky"], AB["0"][2], tol=0.001)
check("sky, 40 off", res[40]["sky"], AB["40"][2], tol=0.001)

fig, ax = plt.subplots(figsize=(7, 3.2))
ks = list(res)
ax.plot(ks, [100 * res[k]["tree"] / base for k in ks], "o-", color=INK, label="the best tree neurons")
ax.plot([0, 40], [100, 100 * rnd40["tree"] / base], "o--", color=INK2, mfc=PAPER, label="random neurons")
ax.set_xlabel("neurons switched off, of 512"); ax.set_ylabel("trees left (%)"); ax.set_ylim(0, 110); ax.legend()
plt.show()

Thirty-two neurons of 512 hold four-fifths of the trees. Where the trees were, the painter puts building and sky: it leaves no hole.

## 5. Painting trees, and the painter's rules (Plates VIII–IX)
Now the reverse: switch the 20 best tree neurons *on*, at 3× their 99th percentile, in one 4 × 4-cell patch of b32,
at 16 places on a 4 × 4 grid. Measure the tree share in the patch and a one-cell margin, before and after (Mask2Former),
and note what the place was before (its commonest label).

In [ ]:
def switch_on(cy, cx, k=20, pk=4, strength=3.0):
    idx = torch.tensor(TREE[:k], device=DEV)
    y0, x0 = max(0, cy - pk // 2), max(0, cx - pk // 2)
    def f(o):
        o = o.clone(); o[:, idx, y0:y0 + pk, x0:x0 + pk] = (strength * Q99[idx])[None, :, None, None]
        return o
    return f

grid = {}
for s in (5019, 5018):
    im0 = paint([s]); sg0 = mask2former(im0)[0]
    for gy in range(4):
        for gx in range(4):
            cy, cx = int((gy + 0.5) * 8), int((gx + 0.5) * 8)
            im1 = paint([s], switch_on(cy, cx)); sg1 = mask2former(im1)[0]
            ys = slice(max(0, (cy - 3) * 8), (cy + 3) * 8); xs = slice(max(0, (cx - 3) * 8), (cx + 3) * 8)
            where = max(("building", "sky", "grass", "tree", "road"), key=lambda w: float((sg0[ys, xs] == MCLS[w]).float().mean()))
            grid[(s, gy, gx)] = (where, float((sg0[ys, xs] == MCLS["tree"]).float().mean()), float((sg1[ys, xs] == MCLS["tree"]).float().mean()), im1[0])
PAINT = {"5019_g00": ["sky", 0.0, 0.887], "5019_g12": ["building", 0.148, 0.058], "5018_g31": ["building", 0.151, 0.884], "5018_g00": ["sky", 0.085, 0.97]}
for key, (w, b4, af) in PAINT.items():
    s, g = key.split("_")
    gw, gb, ga, _ = grid[(int(s), int(g[1]), int(g[2]))]
    print(f"  church {s}, place {g}: was {gw}; tree {100 * gb:.0f}% -> {100 * ga:.0f}%")
    check(f"church {s}, place {g}: what it was", gw, w)
    check(f"  tree after", ga, af, tol=0.03)
works = lambda v: v[2] >= 0.5 and v[2] - v[1] >= 0.25
COUNTS = {"sky": [7, 9], "building": [2, 9], "tree": [2, 14]}
for w in ("sky", "building"):
    pl = [v for v in grid.values() if v[0] == w]
    print(f"  {w}: {sum(works(v) for v in pl)} of {len(pl)} places gain a tree")
    check(f"{'walls' if w == 'building' else w}: places that gain a tree", sum(works(v) for v in pl), COUNTS[w][0], tol=1)
    check(f"{'walls' if w == 'building' else w}: places of this kind", len(pl), COUNTS[w][1], tol=1)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9.5, 3.4))
axs[0].imshow(show(paint([5019])[0])); axs[0].set_title("church 5019", fontsize=9)
for a, (gy, gx, t_) in zip(axs[1:], ((0, 0, "trees on, in the sky"), (1, 2, "trees on, on the tower"))):
    a.imshow(show(grid[(5019, gy, gx)][3])); a.set_title(t_, fontsize=9)
    cy, cx = int((gy + 0.5) * 8), int((gx + 0.5) * 8)
    a.add_patch(plt.Rectangle(((cx - 2) * 8, (cy - 2) * 8), 32, 32, fill=False, ec=INK, lw=1.8))
for a in axs: a.axis("off")
plt.show()

Into the sky: a tree crown, floating, with no trunk. On the tower: a brown smudge, and less tree than before. The same neurons, switched on the same way, work in the sky but rarely on a wall: the painter has rules about where a tree can go.

## 6. Doors, and a painter that remembers (Plates X–XI)
GAN Dissection's famous demo paints doors. Can we find door neurons here? Section 2 also scored every neuron for doors.

In [ ]:
print("best door IoU: b16", round(D16["door"][0][1], 4), " b32", round(D32["door"][0][1], 4))
check("b16: best door IoU", D16["door"][0][1], [0.002, 0.002][0], tol=0.001)
check("b32: best door IoU", D32["door"][0][1], [0.002, 0.002][1], tol=0.001)
check("no neuron with door IoU over 0.05, either block", D16["door_n05"] + D32["door_n05"], 0)

No door neurons: the segmenter barely sees doors in these small churches (0.004% of pixels), and no neuron's map matches
them. (The lecture also asked CLIP which neurons draw doors and switched those on: a wall's chance of looking like a door
rose only from 0.09 to 0.14.)

**The watermark.** Some churches come out with a stock-photo watermark painted in, copied from the photos the painter
learned from. Subtracting a blurred copy makes faint overlaid text stand out.

In [ ]:
from scipy.ndimage import gaussian_filter
im = show(paint([8171])[0]); g = im.mean(2)
hp = np.clip(0.5 + 6 * (g - gaussian_filter(g, 2.5)), 0, 1)
fig, axs = plt.subplots(1, 2, figsize=(7.5, 3.8))
axs[0].imshow(im); axs[0].set_title("seed 8171, as painted", fontsize=9)
axs[1].imshow(hp, cmap="gray"); axs[1].set_title("minus a blurred copy", fontsize=9)
for a in axs: a.axis("off")
plt.show()

## Your turn

1. **Another church.** Switch off 32 tree neurons on church 5164. How much tree is left? *Hint: section 3's loop.*
2. **The other block.** Switch off b16's 32 best tree neurons on the 200 test churches. More or fewer trees left than
   with b32? *Hint: `shares(zero(...), layer="b16.conv1")`, with b16's own tree neurons from `D16`.*
3. **Beside the trees.** On church 5018, switch the tree neurons on at place g31 (row 3, column 1), a wall next to
   trees. *Hint: `grid`.*
4. **The sky neuron.** Which b32 neuron matches the sky best, and how well? *Hint: `D32["sky"]`.*

In [ ]:
# Your turn 1

In [ ]:
# Your turn 2

In [ ]:
# Your turn 3

In [ ]:
# Your turn 4

## Solutions

In [ ]:
# 1. Another church
v = share(mask2former(paint([5164], zero(TREE[:32]))), "tree")
print(f"church 5164, 32 tree neurons off: tree {100 * v:.1f}%")
check("church 5164, 32 off: tree share", v, 0.0038, tol=0.002)

In [ ]:
# 2. The other block
T16 = [u for u, _ in D16["tree"]]
r16 = shares(zero(T16[:32]), layer="b16.conv1")
print(f"b16, 32 off: trees left {100 * r16['tree'] / base:.0f}%  (b32: {100 * res[32]['tree'] / base:.0f}%)")
check("b16, 32 tree neurons off: tree share", r16["tree"], 0.0128, tol=0.0006)
# b16 removes even more: its neurons are less sharp, but the trees are held by few of them.

In [ ]:
# 3. Beside the trees
w, b4, af, _ = grid[(5018, 3, 1)]
print(f"church 5018, g31: was {w}; tree {100 * b4:.0f}% -> {100 * af:.0f}%")
check("church 5018, g31: tree after", af, {"5019_g00": ["sky", 0.0, 0.887], "5019_g12": ["building", 0.148, 0.058], "5018_g31": ["building", 0.151, 0.884], "5018_g00": ["sky", 0.085, 0.97]}["5018_g31"][2], tol=0.03)
# A wall takes a tree when trees are already there: the painter extends them.

In [ ]:
# 4. The sky neuron
print("b32's best sky neurons:", [(u, round(v, 3)) for u, v in D32["sky"][:3]])
check("b32's best sky neuron", D32["sky"][0][0], [336, 0.139][0])
check("its IoU with the sky", D32["sky"][0][1], [336, 0.139][1], tol=0.002)
print(f"whole notebook: {(time.time() - T0) / 60:.1f} minutes")